In [1]:
import pickle
import pandas as pd

with open("titanic_step4_importance_train.pickle", "rb") as pickle_filename :
    train_importance = pickle.load(pickle_filename)
with open("titanic_step4_importance_test.pickle", "rb") as pickle_filename :
    test_importance = pickle.load(pickle_filename)
with open("titanic_step4_importance_train_y.pickle", "rb") as pickle_filename :
    train_answer = pickle.load(pickle_filename)

In [2]:
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from scipy import stats

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [3]:
hyperparams = {
    "C": stats.uniform(0, 50), # 하이퍼 파라미터 값을 결정, 4, 13, 20 -> 크면 : 최대한 정확도 근접 (규제 강도 높인다) : 과적합 위험, 작으면 : 대충, 대략
    "gamma": stats.uniform(0, 1)
}

gd = RandomizedSearchCV(
    estimator=SVC(random_state=1),
    param_distributions=hyperparams,
    n_iter=100,
    cv=5,
    scoring="accuracy",
    random_state=1,
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

df = pd.DataFrame(gd.cv_results_)
print(df[["params", "mean_test_score"]])

0.8397448105122833
{'C': np.float64(2.227593927238086), 'gamma': np.float64(0.1074941291060929)}
                                               params  mean_test_score
0   {'C': 20.8511002351287, 'gamma': 0.72032449344...         0.811560
1   {'C': 0.005718740867244332, 'gamma': 0.3023325...         0.619641
2   {'C': 7.337794540855652, 'gamma': 0.0923385947...         0.832991
3   {'C': 9.313010568883545, 'gamma': 0.3455607270...         0.818308
4   {'C': 19.838373711533496, 'gamma': 0.538816734...         0.820574
..                                                ...              ...
95  {'C': 13.164838524355549, 'gamma': 0.065961090...         0.834120
96  {'C': 36.753298164433474, 'gamma': 0.772178029...         0.809306
97  {'C': 45.3907926251762, 'gamma': 0.93197206919...         0.807046
98  {'C': 0.6975786487798508, 'gamma': 0.234362086...         0.831829
99  {'C': 30.83891785008288, 'gamma': 0.9490163206...         0.805916

[100 rows x 2 columns]


In [4]:
hyperparams = {
    "C": [10, 15, 20, 23, 25, 30, 50],
    "gamma": [0.001, 0.01, 0.05, 0.06, 0.07, 0.1]
}

gd = GridSearchCV(
    estimator=SVC(random_state=1),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

df = pd.DataFrame(gd.cv_results_)
print(df[["params", "mean_test_score"]])

Fitting 5 folds for each of 42 candidates, totalling 210 fits
0.8352440804926046
{'C': 15, 'gamma': 0.05}
                       params  mean_test_score
0   {'C': 10, 'gamma': 0.001}         0.791164
1    {'C': 10, 'gamma': 0.01}         0.829563
2    {'C': 10, 'gamma': 0.05}         0.831854
3    {'C': 10, 'gamma': 0.06}         0.834114
4    {'C': 10, 'gamma': 0.07}         0.832984
5     {'C': 10, 'gamma': 0.1}         0.831861
6   {'C': 15, 'gamma': 0.001}         0.795664
7    {'C': 15, 'gamma': 0.01}         0.826185
8    {'C': 15, 'gamma': 0.05}         0.835244
9    {'C': 15, 'gamma': 0.06}         0.829607
10   {'C': 15, 'gamma': 0.07}         0.831854
11    {'C': 15, 'gamma': 0.1}         0.827347
12  {'C': 20, 'gamma': 0.001}         0.802444
13   {'C': 20, 'gamma': 0.01}         0.827315
14   {'C': 20, 'gamma': 0.05}         0.827347
15   {'C': 20, 'gamma': 0.06}         0.829594
16   {'C': 20, 'gamma': 0.07}         0.830718
17    {'C': 20, 'gamma': 0.1}         0.823964
1

In [5]:
learning_rate = [0.01, 0.05, 0.1, 0.2] # 만약 최초 모델을 통해서 학습 -> 오류 -> 어느정도 범위까지 줄여나가게 할 것인가? = 학습률
n_estimators = [100, 1000, 2000]
max_depth = [3, 5, 10, 15]

hyerparams = {
    "learning_rate": learning_rate,
    "n_estimators": n_estimators,
    "max_depth": max_depth
}

gd = GridSearchCV(
    estimator= GradientBoostingClassifier(random_state=1),
    param_grid=hyerparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
0.8431282930235511
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 1000}


In [19]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

hyperparams = {
    "penalty": ["l1", "l2", "elasticnet"],
    "C": stats.uniform(0, 1000)
}

gd = RandomizedSearchCV(
    estimator= LogisticRegression(random_state=1, solver='lbfgs', max_iter=1000),
    param_distributions=hyperparams,
    n_iter=100,
    cv=5,
    scoring="accuracy",
    random_state=1,
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

0.8409064940011428
{'C': np.float64(491.5731592803383), 'penalty': 'l2'}


In [7]:
np.linspace(700, 900, 10)

array([700.        , 722.22222222, 744.44444444, 766.66666667,
       788.88888889, 811.11111111, 833.33333333, 855.55555556,
       877.77777778, 900.        ])

In [8]:
penalty = ["l1", "l2"]
C = np.linspace(700, 900, 200)

hyperparams = {
    "penalty": penalty,
    "C": C
}

gd = GridSearchCV(
    estimator=LogisticRegression(random_state=1),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 400 candidates, totalling 2000 fits
0.8409064940011426
{'C': np.float64(706.0301507537688), 'penalty': 'l2'}


In [20]:
!pip install bayesian-optimization

Defaulting to user installation because normal site-packages is not writeable


In [9]:
import numpy as np
from xgboost import XGBClassifier
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score

pbounds = {
    "learning_rate": (0.01, 0.5), # 하나의 트리가 다음번째 트리에 얼마나 강하게 반영이 될지 결정하는 속성
    "n_estimators": (100, 1000), # 몇 개의 트리를 만들것인가?
    "max_depth": (3, 10), # XGBoost -> 3~6적합
    "min_child_weight": (0, 10), # 트리 내 질문을 하려고 할 때, 샘플 데이터 양
    "subsample": (0.5, 1.0), # 각 트리를 학습할 때, 사용하는 데이터 비율
    "colsample_bytree": (0.5, 1.0), # 학습시키고자 하는 데이터의 컬럼을 전체다 사용? 일부만 사용?
    "gamma": (0, 5) # 성능 개선의 최대치
}

def xgbost_hyper_param(learning_rate, n_estimators, max_depth, min_child_weight, subsample, colsample_bytree, gamma) :
    max_depth = int(max_depth)
    n_estimators = int(n_estimators)
    
    clf = XGBClassifier(
        max_depth = max_depth,
        min_child_weight = min_child_weight,
        learning_rate = learning_rate,
        n_estimators = n_estimators,
        subsample = subsample,
        colsample_bytree = colsample_bytree,
        gamma = gamma,
        random_state = 1,
        eval_metric = "logloss"
    )
    return np.mean(cross_val_score(clf, train_importance, train_answer, cv=5, scoring="accuracy"))

optimizer = BayesianOptimization(f=xgbost_hyper_param, pbounds=pbounds, random_state=1)
optimizer.maximize(init_points=10, n_iter=100)

|   iter    |  target   | learni... | n_esti... | max_depth | min_ch... | subsample | colsam... |   gamma   |
-------------------------------------------------------------------------------------------------------------
| 1         | 0.8194375 | 0.2143407 | 748.29204 | 3.0008006 | 3.0233257 | 0.5733779 | 0.5461692 | 0.9313010 |
| 2         | 0.8182822 | 0.1793247 | 457.09072 | 6.7717171 | 4.1919451 | 0.8426097 | 0.6022261 | 4.3905871 |
| 3         | 0.8070335 | 0.0234199 | 703.42075 | 5.9211336 | 5.5868982 | 0.5701934 | 0.5990507 | 4.0037228 |
| 4         | 0.8216784 | 0.4844481 | 382.08176 | 7.8462583 | 8.7638915 | 0.9473033 | 0.5425221 | 0.1952739 |
| 5         | 0.8115089 | 0.0932169 | 890.32825 | 3.6884278 | 4.2110762 | 0.9789447 | 0.7665826 | 3.4593855 |
| 6         | 0.8318161 | 0.1646026 | 717.85083 | 8.8423797 | 0.1828827 | 0.8750721 | 0.9944305 | 3.7408282 |
| 7         | 0.8295816 | 0.1474175 | 810.35139 | 3.7225820 | 4.4789352 | 0.9542977 | 0.6468070 | 1.4388766 |
| 8       

In [7]:
import sys
!{sys.executable} -m pip install bayesian-optimization

  Using cached bayesian_optimization-3.1.0-py3-none-any.whl.metadata (11 kB)
Using cached bayesian_optimization-3.1.0-py3-none-any.whl (36 kB)


In [10]:
optimizer.max

{'target': np.float64(0.8419983495207262),
 'params': {'learning_rate': np.float64(0.01),
  'n_estimators': np.float64(113.82657767044938),
  'max_depth': np.float64(6.5317722366886395),
  'min_child_weight': np.float64(0.0),
  'subsample': np.float64(1.0),
  'colsample_bytree': np.float64(0.7924265863656789),
  'gamma': np.float64(0.0)}}

In [11]:
learning_rate = [0.001, 0.005, 0.01, 0.05, 0.06, 0.1, 0.12, 0.15, 0.17, 0.2]
n_estimators = [10, 50, 60, 75, 85, 100, 125, 150, 200, 250, 500, 1000]

hyperparams = {
    "learning_rate": learning_rate,
    "n_estimators": n_estimators
}

gd = GridSearchCV(
    estimator = XGBClassifier(random_state=1, eval_metric="logloss"),
    param_grid = hyperparams,
    verbose = True,
    cv = 5,
    scoring = "accuracy",
    n_jobs = -1
)

gd.fit(train_importance, train_answer)
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 120 candidates, totalling 600 fits
0.83747222751222
{'learning_rate': 0.17, 'n_estimators': 10}


In [12]:
max_depth = [3, 4, 5, 6, 7, 8, 9, 10]
min_child_weight = [1, 2, 3, 4, 5, 6, 7]

hyperparams = {
    "max_depth": max_depth,
    "min_child_weight": min_child_weight
}

gd = GridSearchCV(
    estimator=XGBClassifier(learning_rate=0.17, n_estimators=10, random_state=1, eval_metric="logloss"),
    param_grid=hyperparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer)
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 56 candidates, totalling 280 fits
0.83747222751222
{'max_depth': 6, 'min_child_weight': 1}


In [13]:
gamma=[i * 0.1 for i in range(0, 5)] # 표준 손실 감소량 => 1에 가까울 수록 노드를 쪼개는데 비해서 그만큼 현격한 결과가 도출될 때에만 쪼개라.
subsample=[0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1] # 학습시키고자 하는 데이터의 전체 행을 기준으로 몇 프로
colsample_bytree=[0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1] # 학습시키고자 하는 데이터의 전체 열을 기준으로 몇 프로
reg_alpha=[1e-5, 1e-2, 0.1, 1, 100] # 학습시키고자 하는 데이터의 결과값 너무 작은, 큰 값에 치우지지 않게 하기 위한 옵션

hyerparams={
    "gamma": gamma,
    "subsample": subsample,
    "colsample_bytree": colsample_bytree,
    "reg_alpha": reg_alpha
}

gd=GridSearchCV(
    estimator=XGBClassifier(
        learning_rate=0.17,
        n_estimators=10,
        max_depth=6,
        min_child_weight=1,
        random_state=1,
        eval_metric="logloss"
    ),
    param_grid=hyerparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer)
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 2025 candidates, totalling 10125 fits
0.8476163270488162
{'colsample_bytree': 0.85, 'gamma': 0.2, 'reg_alpha': 1e-05, 'subsample': 1}


In [16]:
# RandomForest

In [15]:
n_estimators = [10, 50, 100, 200]
max_depth = [3, None]
max_features = [0.1, 0.2, 0.5, 0.8, "sqrt", "log2"]
min_samples_split = [2, 4, 6, 8, 10] # 의사결정 트리에서 분기해도되는지에 대한 여부 => 최소값
min_samples_leaf = [2, 4, 6, 8, 10] # 의사결정 트리로 분기가 된 이후에 만들어지는 노드 값의 최소값

hyerparams={
    "n_estimators": n_estimators,
    "max_depth": max_depth,
    "max_features": max_features,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf
}

gd = GridSearchCV(
    estimator=RandomForestClassifier(random_state=1),
    param_grid=hyerparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 1200 candidates, totalling 6000 fits
0.8487526185488479
{'max_depth': None, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 8, 'n_estimators': 200}


In [17]:
# Extra Trees

In [18]:
n_estimators=[10, 50, 100, 200]
max_depth=[3, None]
max_features = [0.1, 0.2, 0.5, 0.8, "sqrt", "log2"]
min_samples_split = [2, 4, 6, 8, 10] # 의사결정 트리에서 분기해도되는지에 대한 여부 => 최소값
min_samples_leaf = [2, 4, 6, 8, 10] # 의사결정 트리로 분기가 된 이후에 만들어지는 노드 값의 최소값

hyerparams={
    "n_estimators": n_estimators,
    "max_depth": max_depth,
    "max_features": max_features,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf
}

gd = GridSearchCV(
    estimator=ExtraTreesClassifier(random_state=1),
    param_grid=hyerparams,
    verbose=True,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

gd.fit(train_importance, train_answer.values.ravel())
print(gd.best_score_)
print(gd.best_params_)

Fitting 5 folds for each of 1200 candidates, totalling 6000 fits
0.8532787405573542
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 8, 'n_estimators': 10}


In [21]:
logreg_model = LogisticRegression(C=706, penalty="l2", random_state=1)

xgb_model = XGBClassifier(
    eval_metric = "logloss",
    learning_rate = 0.01,
    n_estimators = 113,
    max_depth = 6,
    min_child_weight = 0.0,
    gamma = 0.0,
    random_state = 1
)

random_model = RandomForestClassifier(
    max_depth = None,
    max_features = 0.8,
    min_samples_leaf = 2,
    min_samples_split = 8,
    n_estimators = 200,
    random_state = 1
)

extra_model = ExtraTreesClassifier(
    max_depth = None,
    max_features = 'sqrt',
    min_samples_leaf = 2,
    min_samples_split = 8,
    n_estimators = 10,
    random_state = 1
)

In [22]:
from sklearn.ensemble import VotingClassifier

grid_hard = VotingClassifier(
    estimators=[
        ("Logistic Regression", logreg_model),
        ("XGBoost", xgb_model),
        ("Random Forest", random_model),
        ("Extra Trees", extra_model),
    ], voting="hard"
)

score = cross_val_score(grid_hard, train_importance, train_answer, cv=5, scoring="accuracy")
print(np.mean(score) * 100)

85.55386275630038


In [23]:
from sklearn.ensemble import VotingClassifier

grid_soft = VotingClassifier(
    estimators=[
        ("Logistic Regression", logreg_model),
        ("XGBoost", xgb_model),
        ("Random Forest", random_model),
        ("Extra Trees", extra_model),
    ], voting="soft"
)

score = cross_val_score(grid_soft, train_importance, train_answer, cv=5, scoring="accuracy")
print(np.mean(score) * 100)

84.08557100234877


In [26]:
test = pd.read_csv("titanic/test.csv")
submission = pd.DataFrame(columns=["PassengerId", "Survived"])
submission["PassengerId"] = test["PassengerId"]

In [27]:
submission

,PassengerId,Survived
0,892,NaN
1,893,NaN
2,894,NaN
3,895,NaN
4,896,NaN
...,...,...
413,1305,NaN
414,1306,NaN
415,1307,NaN
416,1308,NaN


In [28]:
grid_hard.fit(train_importance, train_answer)

VotingClassifier(estimators=[('Logistic Regression',
                              LogisticRegression(C=706, random_state=1)),
                             ('XGBoost',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='logloss',
                                            feature_types=None,
                                            feature_weights=...
                                            min_child_weight=0.0, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=113, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                             ('Random Forest',
                              RandomForestClassifier(max_features=0.8,
                                                     min_samples_leaf=2,
                                                     min_samples_split=8,
                                                     n_estimators=200,
                                                     random_state=1)),
                             ('Extra Trees',
                              ExtraTreesClassifier(min_samples_leaf=2,
                                                   min_samples_split=8,
                                                   n_estimators=10,
                                                   random_state=1))])

In [29]:
submission["Survived"]=grid_hard.predict(test_importance)
submission = submission.astype("int")
submission.head()

ValueError: Length of values (416) does not match length of index (418)

In [30]:
len(submission)

418

In [31]:
len(test_importance)

416

In [33]:
test_importance

,Initial_0,Pclass_2,Sex_0,Sex_1,Cabin_0,Initial_6,Fare_0,LowChance_0,HighChance_0,LowChance_1,...,Age_6,Family_0,Ticket_Num_Cut_3,Fare_3,Cabin_4,Ticket_Initial2_2,Cabin_1,Age_1,Ticket_Initial2_15,Family_4
0,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
1,0,1,0,1,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,1,0,1,0,1,1,...,1,1,0,0,0,0,0,0,0,0
3,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
4,0,1,0,1,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,0,1,0,1,1,0,1,1,1,0,...,0,1,0,0,0,0,0,0,0,0
413,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
415,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
416,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0


In [32]:
submission

,PassengerId,Survived
0,892,NaN
1,893,NaN
2,894,NaN
3,895,NaN
4,896,NaN
...,...,...
413,1305,NaN
414,1306,NaN
415,1307,NaN
416,1308,NaN


In [34]:
submission = submission.loc[test_importance.index]
submission["Survived"] = grid_hard.predict(test_importance)
submission = submission.astype("int")

In [35]:
submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [36]:
submission.to_csv("titanic_predict.csv", header=True, index=False)

In [37]:
import os
os.environ["KAGGLE_API_TOKEN"] = ""

!kaggle competitions list

/Users/admin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
ref                                                                                 deadline             category                reward  teamCount  userHasEntered  
----------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3       2026-04-15 23:59:00  Featured         2,207,152 Usd       1032           False  
https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection            2026-02-13 23:59:00  Research           200,000 Usd        476           False  
https://www.kaggle.com/competitions/google-tunix-hackathon          

In [38]:
!kaggle competitions submit -c titanic -f titanic_predict.csv -m "Test"

/Users/admin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
100%|██████████████████████████████████████| 2.76k/2.76k [00:00<00:00, 3.09kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster

In [ ]:
kaggle competitions submit -c titanic -f submission.csv -m "Message"

In [39]:
test_ids = set(test["PassengerId"])

In [41]:
test_importance

,Initial_0,Pclass_2,Sex_0,Sex_1,Cabin_0,Initial_6,Fare_0,LowChance_0,HighChance_0,LowChance_1,...,Age_6,Family_0,Ticket_Num_Cut_3,Fare_3,Cabin_4,Ticket_Initial2_2,Cabin_1,Age_1,Ticket_Initial2_15,Family_4
0,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
1,0,1,0,1,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,1,0,1,0,1,1,...,1,1,0,0,0,0,0,0,0,0
3,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
4,0,1,0,1,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
412,0,1,0,1,1,0,1,1,1,0,...,0,1,0,0,0,0,0,0,0,0
413,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
415,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0
416,1,1,1,0,1,0,1,0,1,0,...,0,1,0,0,0,0,0,0,0,0


In [40]:
mp_ids = set(test_importance["PassengerId"])

KeyError: 'PassengerId'

In [42]:
missing_idx = test.index.difference(test_importance.index)
missing_idx, len(missing_idx)

(Index([88, 414], dtype='int64'), 2)

In [43]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": 0
}, index=test.index)

pred = grid_hard.predict(test_importance)

submission.loc[test_importance.index, "Survived"] = pred
submission["Survived"] = submission["Survived"].astype(int)

len(submission), submission["Survived"].isna().sum()

(418, np.int64(0))

In [48]:
submission.to_csv("titanic_predict.csv", header=True, index=False)

In [49]:
len(submission)

418

In [50]:
!kaggle competitions submit -c titanic -f titanic_predict.csv -m "Test"

/Users/admin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
100%|██████████████████████████████████████| 2.77k/2.77k [00:00<00:00, 2.93kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster